# Gọi lại invalid >40% — giữ nguyên prompt context 8.192 byte

Chỉ chạy **6 nhánh, 6.322 target originally invalid**. Một cell/nhánh gọi lại prompt, checkpoint, ghép với các response valid cũ, chạy evaluator trên toàn cohort và cập nhật cả SHARE lẫn run gốc ở C.

**Model/provider theo lựa chọn hiện tại:** FPT AI Factory / `DeepSeek-V4-Flash`, lấy credential từ `D:/JUNIOR/LabThayTho/.env` (không ghi key vào notebook/log). Giữ prompt nguyên văn, tối đa **8.192 UTF-8 byte**, temperature0, seed42, max_tokens256. Không dùng contract 24.000 byte và không dựng lại prompt.

Kết quả ghép là **mixed backbone**: phần valid cũ là Qwen/Vertex, phần gọi lại là FPT/DeepSeek. Notebook ghi `repair_manifest.json`, không trình bày như benchmark cùng một backbone. Snapshot gốc còn trong Git; notebook cũng giữ baseline tại `_rerun_invalid_over40_context8192_20261020/baseline` để resume và đối soát.

Chạy cell setup, rồi chạy từng cell nhánh hoặc Run All. Mỗi target thử tối đa5 lần kể cả response rỗng/sai0/1; dừng khi hợp lệ. Tổng tối đa31.610 HTTP attempts nếu tất cả dùng hết5 lượt. Retry không đổi prompt/model/output budget. Setup chỉ đọc/kiểm tra và sao lưu, chưa gọi API.

Artifact cập nhật ở mỗi nhánh: `inference/raw_responses.jsonl`, `evaluation/predictions.jsonl`, `evaluation/evaluation.json`, `repair_manifest.json`; các completion/ablation summary có sẵn cũng được cập nhật. Prompt/retrieval và preparation freeze giữ nguyên; repair manifest mô tả phần inference đã thay. `attempts.jsonl` lưu response provider, finish_reason, usage và error của từng lần gọi.

Evaluator nhúng nguyên logic hiện tại đã thống nhất giữa Devign/Reveal, gồm hai baseline và delta. Metric vẫn valid-only, báo coverage/invalid; evaluator chạy trên toàn bộ response đã ghép, không chỉ phần retry. Các bảng ledger/copy_manifest snapshot cũ không tự đổi; dùng evaluation mới và repair manifest để báo kết quả sau retry.

Resume: chạy lại setup/cell nhánh sẽ dùng checkpoint, không gọi lại target đã hoàn thành hoặc đã hết5 attempts. Nếu kernel chết cứng để lại `running.lock`, kiểm tra không còn notebook/kernel chạy nhánh đó rồi xóa đúng file lock được báo; không xóa checkpoint.

Vị trí notebook: `C:/MY-FILES/LAB-THAY-THO/Graduation-Project/`. SHARE, `.env` và checkpoint/snapshot tiếp tục ở ổ D; run gốc ổ C được tính từ `PROJECT_ROOT / Ablation_V2`.


In [ ]:
import copy
import hashlib
import json
import math
import os
import shutil
import tempfile
import threading
import time
import urllib.error
import urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path('C:/MY-FILES/LAB-THAY-THO/Graduation-Project')
DATA_ROOT = Path('D:/JUNIOR/LabThayTho')
SHARE = DATA_ROOT / 'SHARE_36_RUNS_CONTEXT8192_20261020_205918'
ORIGINAL = PROJECT_ROOT / 'Ablation_V2'
WORK = DATA_ROOT / '_rerun_invalid_over40_context8192_20261020'
ENV_FILE = DATA_ROOT / '.env'
MODEL = 'DeepSeek-V4-Flash'
BASE_URL = 'https://token-api.fpt.ai/v1'
PROVIDER = 'FPT AI Factory'
CONTEXT_BYTES = 8192
MAX_TOKENS = 256
TEMPERATURE = 0.0
SEED = 42
MAX_ATTEMPTS = 5
WORKERS = 4
REQUEST_TIMEOUT = 180
BRANCHES = [
    'Devign/Ablations/Plus-C8-mmr-selection/test_v2',
    'Reveal/Ablations/No-C2-stratified-demo-selection/test_v2',
    'Reveal/Ablations/No-C3-GEPA-prompt-optimization/test_v2',
    'Reveal/Ablations/No-C5-graph-verbalization/test_v2',
    'Reveal/Ablations/Plus-C1-tri-signal-retrieval/test_v2',
    'Reveal/Ablations/Plus-C7-security-cot/test_v2',
]
FILES = ('inference/raw_responses.jsonl', 'evaluation/predictions.jsonl',
         'evaluation/evaluation.json', 'prompts/prompts.jsonl', 'run_manifest.json', 'freeze.json')
_log_lock = threading.Lock()


def now():
    return datetime.now(timezone.utc).isoformat()


def sha_bytes(value):
    return hashlib.sha256(value).hexdigest()


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as stream:
        return [json.loads(line) for line in stream if line.strip()]


def atomic_bytes(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('wb', dir=path.parent, delete=False) as stream:
        temporary = Path(stream.name)
        stream.write(data)
        stream.flush()
        os.fsync(stream.fileno())
    try:
        os.replace(temporary, path)
    finally:
        if temporary.exists():
            temporary.unlink()


def write_json(path, value):
    atomic_bytes(path, (json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + '\n').encode('utf-8'))


def write_jsonl(path, rows):
    atomic_bytes(path, ''.join(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in rows).encode('utf-8'))


def index_unique(rows, name):
    result = {}
    for row in rows:
        rid = row['record_id']
        if rid in result:
            raise ValueError(f'Duplicate ID in {name}: {rid}')
        result[rid] = row
    return result


def load_settings():
    values = {}
    for line in ENV_FILE.read_text(encoding='utf-8-sig').splitlines():
        key, separator, value = line.strip().removeprefix('export ').partition('=')
        if not separator or key.strip() not in {'FPT_API', 'FPT_MODEL', 'FPT_BASE_URL'}:
            continue
        value = value.strip()
        if value[:1] in {chr(39), chr(34)}:
            quote = value[0]
            end = value.find(quote, 1)
            value = value[1:end] if end >= 0 else value[1:]
        else:
            value = value.split(' #', 1)[0].strip()
        values[key.strip()] = value
    if not values.get('FPT_API'):
        raise RuntimeError(f'Missing FPT_API in {ENV_FILE}; no provider fallback')
    model = values.get('FPT_MODEL')
    endpoint = (values.get('FPT_BASE_URL') or 'https://mkp-api.fptcloud.com/v1').rstrip('/')
    if model != MODEL or endpoint != BASE_URL:
        raise RuntimeError('FPT model/endpoint changed. Review MODEL/BASE_URL and use a new WORK directory.')
    return values['FPT_API']


# The exact current evaluator implementation is embedded below, independent
# of graph/ML imports and independent of the current 24000-byte contract.
_EVALUATOR_SOURCE = 'def parse_prediction(raw: str) -> int | None:\n    lines = [line.strip() for line in str(raw).splitlines() if line.strip()]\n    if not lines or lines[-1] not in {"0", "1"}:\n        return None\n    return int(lines[-1])\n\ndef ratio(num: int, den: int) -> float | None:\n    return None if den == 0 else num / den\n\ndef evaluate(response_path: Path, labels_path: Path, output: Path) -> dict[str, Any]:\n    labels = {row["record_id"]: int(row["sealed_label"]) for row in read_jsonl(labels_path)}\n    responses = read_jsonl(response_path)\n    predictions: list[dict[str, Any]] = []\n    valid = []\n    invalid = 0\n    for row in responses:\n        if row["record_id"] not in labels:\n            raise ValueError(f"Response has no sealed label: {row[\'record_id\']}")\n        prediction = parse_prediction(row.get("raw_response", ""))\n        item = {"record_id": row["record_id"], "sealed_label": labels[row["record_id"]], "prediction": prediction, "raw_response": row.get("raw_response", ""), "request_status": row.get("request_status")}\n        predictions.append(item)\n        if prediction is None:\n            invalid += 1\n        else:\n            valid.append(item)\n    def score(items):\n        tp = sum(item["prediction"] == 1 and item["sealed_label"] == 1 for item in items)\n        tn = sum(item["prediction"] == 0 and item["sealed_label"] == 0 for item in items)\n        fp = sum(item["prediction"] == 1 and item["sealed_label"] == 0 for item in items)\n        fn = sum(item["prediction"] == 0 and item["sealed_label"] == 1 for item in items)\n        recall = ratio(tp, tp + fn)\n        specificity = ratio(tn, tn + fp)\n        denominator = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))\n        # Undefined precision/F1/MCC are zero for a nonempty population;\n        # an empty population has no metrics. Missing-class recall/BA stay null.\n        return {\n            "n": len(items),\n            "confusion_matrix": {"tp": tp, "tn": tn, "fp": fp, "fn": fn},\n            "accuracy_valid": ratio(tp + tn, len(items)),\n            "precision": (ratio(tp, tp + fp) if tp + fp else (0.0 if items else None)),\n            "recall": recall,\n            "f1": (ratio(2 * tp, 2 * tp + fp + fn) if 2 * tp + fp + fn else (0.0 if items else None)),\n            "balanced_accuracy": None if recall is None or specificity is None else (recall + specificity) / 2,\n            "mcc": (tp * tn - fp * fn) / denominator if denominator else (0.0 if items else None),\n            "predicted_vulnerable_rate_valid": ratio(tp + fp, len(items)),\n        }\n\n    observed = score(valid)\n    baselines = {\n        "always_vulnerable": score([{**item, "prediction": 1} for item in valid]),\n        "always_non_vulnerable": score([{**item, "prediction": 0} for item in valid]),\n    }\n    comparison_keys = ("accuracy_valid", "precision", "recall", "f1", "mcc", "balanced_accuracy")\n    metrics = {\n        **observed,\n        "n_responses": len(responses), "n_valid": len(valid), "n_invalid": invalid,\n        "invalid_rate": ratio(invalid, len(responses)),\n        "metric_population": "valid_predictions_only",\n        "coverage": ratio(len(valid), len(responses)),\n        "undefined_metric_policy": "nonempty: undefined precision/F1/MCC=0; missing-class recall/balanced_accuracy=null; empty: all metrics=null",\n        **baselines,\n        "delta_vs_trivial_baselines": {\n            name: {key: None if observed[key] is None or baseline[key] is None else observed[key] - baseline[key]\n                   for key in comparison_keys}\n            for name, baseline in baselines.items()\n        },\n    }\n    write_jsonl(output / "predictions.jsonl", predictions)\n    write_json(output / "evaluation.json", metrics)\n    return metrics'
exec(compile(_EVALUATOR_SOURCE, '<frozen_standard_evaluator>', 'exec'), globals())
EVALUATOR_SHA256 = sha_bytes(_EVALUATOR_SOURCE.encode('utf-8'))


def generation_spec():
    return {'provider': PROVIDER, 'model': MODEL, 'base_url': BASE_URL,
            'context_limit_utf8_bytes': CONTEXT_BYTES, 'temperature': TEMPERATURE,
            'seed': SEED, 'max_tokens': MAX_TOKENS, 'thinking': False,
            'max_attempts_total_per_invalid_target': MAX_ATTEMPTS,
            'retry_request_errors_and_invalid_content': True,
            'evaluator_sha256': EVALUATOR_SHA256}


def preflight():
    WORK.mkdir(parents=True, exist_ok=True)
    plan_path = WORK / 'plan.json'
    spec = generation_spec()
    if plan_path.exists():
        plan = json.loads(plan_path.read_text(encoding='utf-8'))
        if plan['generation_spec'] != spec or plan['branches'] != BRANCHES:
            raise RuntimeError('Existing repair plan has a different model/config/branch list')
        print('Resuming frozen repair plan:', plan_path)
        return plan
    entries = json.loads((SHARE / 'bang_quan_ly_run.json').read_text(encoding='utf-8'))
    selected = {e['run_id'] for e in entries if e['n_invalid'] / e['n_total'] > .40}
    if selected != set(BRANCHES):
        raise RuntimeError(f'Strict >40% branch selection differs: {sorted(selected)}')
    plan = {'schema': 'invalid_retry_plan_v1', 'created_at': now(), 'branches': BRANCHES,
            'generation_spec': spec, 'runs': {}, 'mixed_backbone': True,
            'benchmark_comparable_single_backbone': False}
    for run_id in BRANCHES:
        src = SHARE / 'runs' / run_id
        dst = ORIGINAL / run_id
        snapshot = WORK / 'baseline' / run_id
        fingerprints = {}
        for name in FILES:
            shash, chash = sha_file(src / name), sha_file(dst / name)
            if shash != chash:
                raise RuntimeError(f'SHARE and C original differ before repair: {run_id}/{name}')
            fingerprints[name] = shash
        raw = read_jsonl(src / 'inference/raw_responses.jsonl')
        preds = index_unique(read_jsonl(src / 'evaluation/predictions.jsonl'), 'source predictions')
        prompts = index_unique(read_jsonl(src / 'prompts/prompts.jsonl'), 'source prompts')
        raw_map = index_unique(raw, 'source raw')
        if set(raw_map) != set(preds) or set(raw_map) != set(prompts):
            raise RuntimeError(f'Raw/prediction/prompt ID sets differ: {run_id}')
        invalid_ids = []
        for row in raw:
            rid = row['record_id']
            prediction = parse_prediction(row.get('raw_response', ''))
            if prediction != preds[rid]['prediction']:
                raise RuntimeError(f'Parser differs from saved prediction: {rid}')
            if preds[rid]['sealed_label'] not in (0, 1):
                raise ValueError(f'Invalid ground truth: {rid}')
            prompt = prompts[rid]['prompt']
            if (len(prompt.encode('utf-8')) > CONTEXT_BYTES
                    or sha_bytes(prompt.encode('utf-8')) != prompts[rid]['prompt_sha256']
                    or row['prompt_sha256'] != prompts[rid]['prompt_sha256']
                    or row.get('prompt') != prompt):
                raise RuntimeError(f'Prompt/hash/8192-byte contract mismatch: {rid}')
            if prediction is None:
                invalid_ids.append(rid)
        if len(invalid_ids) / len(raw) <= .4:
            raise RuntimeError(f'Actual invalid rate is not >40%: {run_id}')
        # Preserve the original cohort for resume even after both target copies
        # are overwritten. Existing snapshots must match, never silently replace.
        for name in FILES:
            backup = snapshot / name
            if backup.exists() and sha_file(backup) != fingerprints[name]:
                raise RuntimeError(f'Conflicting baseline snapshot: {backup}')
            if not backup.exists():
                atomic_bytes(backup, (src / name).read_bytes())
        plan['runs'][run_id] = {'n': len(raw), 'invalid_n': len(invalid_ids),
            'invalid_ids': invalid_ids, 'source_fingerprints': fingerprints,
            'snapshot': str(snapshot), 'share': str(src), 'original': str(dst)}
        print(f'{run_id}: {len(invalid_ids)}/{len(raw)} invalid ({len(invalid_ids)/len(raw):.2%})')
    write_json(plan_path, plan)
    write_json(WORK / 'generation_spec.json', spec)
    atomic_bytes(WORK / 'standard_evaluator.py', _EVALUATOR_SOURCE.encode('utf-8'))
    print('Targets to retry:', sum(r['invalid_n'] for r in plan['runs'].values()))
    return plan


def read_attempts(path):
    if not path.exists():
        return {}
    data = path.read_bytes()
    if data and not data.endswith(b'\n'):
        # A process crash can tear the last append. Preserve it for inspection.
        end = data.rfind(b'\n') + 1
        atomic_bytes(path.with_suffix('.torn'), data[end:])
        atomic_bytes(path, data[:end])
        data = data[:end]
    grouped = {}
    for line in data.splitlines():
        row = json.loads(line)
        grouped.setdefault(row['record_id'], []).append(row)
    for rid, attempts in grouped.items():
        if [a['attempt'] for a in attempts] != list(range(1, len(attempts) + 1)):
            raise ValueError(f'Broken attempt order: {rid}')
    return grouped


def append_attempt(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with _log_lock, path.open('ab') as stream:
        stream.write((json.dumps(record, ensure_ascii=False, sort_keys=True) + '\n').encode('utf-8'))
        stream.flush()
        os.fsync(stream.fileno())


def query_target(original_row, history, log_path, api_key, transport=None):
    rid = original_row['record_id']
    if history and (history[-1]['terminal'] or len(history) >= MAX_ATTEMPTS):
        return history[-1]
    for attempt in range(len(history) + 1, MAX_ATTEMPTS + 1):
        if attempt > 1:
            time.sleep(min(120, 5 * 2 ** (attempt - 2)))
        record = {'record_id': rid, 'prompt_sha256': original_row['prompt_sha256'],
                  'attempt': attempt, 'started_at': now(), 'model': MODEL,
                  'provider': PROVIDER, 'raw_response': '', 'request_error': None,
                  'http_status': None, 'provider_response': None}
        started = time.monotonic()
        payload = {'model': MODEL, 'messages': [{'role': 'user', 'content': original_row['prompt']}],
                   'temperature': TEMPERATURE, 'seed': SEED, 'max_tokens': MAX_TOKENS,
                   'stream': False, 'chat_template_kwargs': {'thinking': False}}
        try:
            if transport is not None:
                body = transport(payload)
            else:
                request = urllib.request.Request(BASE_URL + '/chat/completions',
                    data=json.dumps(payload).encode('utf-8'), method='POST',
                    headers={'Authorization': 'Bearer ' + api_key, 'Content-Type': 'application/json'})
                with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT) as response:
                    record['http_status'] = response.status
                    body = json.load(response)
            record['provider_response'] = body
            content = body.get('data', body)
            choices = content.get('choices', [])
            record['finish_reason'] = choices[0].get('finish_reason') if choices else None
            record['usage'] = content.get('usage')
            message = choices[0].get('message', {}) if choices else {}
            text = message.get('content')
            if isinstance(text, list):
                text = ''.join(part.get('text', '') for part in text if isinstance(part, dict))
            record['raw_response'] = text if isinstance(text, str) else ''
            record['request_status'] = 'ok'
        except urllib.error.HTTPError as exc:
            record['http_status'] = exc.code
            record['request_status'] = 'request_error'
            record['request_error'] = ('HTTP ' + str(exc.code) + ': ' + exc.read(8192).decode('utf-8', 'replace')).replace(api_key, '<redacted>')
        except Exception as exc:
            record['request_status'] = 'request_error'
            record['request_error'] = repr(exc).replace(api_key, '<redacted>')
        prediction = parse_prediction(record['raw_response'])
        record['prediction'] = prediction
        record['parse_status'] = 'valid' if prediction is not None else 'invalid'
        record['response_status'] = ('valid_binary' if prediction is not None else
            'request_error' if record['request_error'] else
            'empty_response' if not record['raw_response'].strip() else 'nonbinary_last_line')
        permanent = record['http_status'] in (400, 401, 403, 404)
        record['terminal'] = prediction is not None or attempt >= MAX_ATTEMPTS or permanent
        record['finished_at'] = now()
        record['wall_seconds'] = time.monotonic() - started
        append_attempt(log_path, record)
        if permanent:
            raise RuntimeError(f'Permanent HTTP {record["http_status"]}; fix provider/config before continuing. Attempt saved.')
        if record['terminal']:
            return record
    raise AssertionError('Unreachable retry state')


def run_branch(run_id, transport=None):
    if run_id not in PLAN['runs']:
        raise ValueError('Branch was not selected by the frozen >40% plan')
    # Prevent two kernels from updating the same branch at once. A stale lock
    # is never removed automatically; inspect it after a hard kernel crash.
    task = WORK / 'repairs' / run_id
    task.mkdir(parents=True, exist_ok=True)
    lock = task / 'running.lock'
    fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    os.write(fd, json.dumps({'pid': os.getpid(), 'started_at': now()}).encode())
    os.close(fd)
    try:
        return _run_branch(run_id, task, transport)
    finally:
        lock.unlink(missing_ok=True)


def _run_branch(run_id, task, transport):
    entry = PLAN['runs'][run_id]
    snapshot = Path(entry['snapshot'])
    for name, expected in entry['source_fingerprints'].items():
        if sha_file(snapshot / name) != expected:
            raise RuntimeError(f'Baseline snapshot changed: {run_id}/{name}')
    original_raw = read_jsonl(snapshot / 'inference/raw_responses.jsonl')
    raw_map = index_unique(original_raw, 'baseline raw')
    invalid_ids = entry['invalid_ids']
    log_path = task / 'attempts.jsonl'
    history = read_attempts(log_path)
    if set(history) - set(invalid_ids):
        raise RuntimeError('Attempt log contains a formerly valid or unknown ID')
    # Credential failures must not be silently resumed as successful repairs.
    if any(v[-1].get('http_status') in (400, 401, 403, 404) for v in history.values()):
        raise RuntimeError('Permanent API error recorded; inspect attempts.jsonl and use a new WORK after fixing config')
    todo = [rid for rid in invalid_ids if not history.get(rid) or not history[rid][-1]['terminal']]
    for rid, attempts in history.items():
        if any(a.get('prompt_sha256') != raw_map[rid]['prompt_sha256']
               or a.get('model') != MODEL or a.get('provider') != PROVIDER for a in attempts):
            raise RuntimeError(f'Checkpoint prompt/model/provider mismatch: {rid}')
    print(f'{run_id}: {len(todo)} remaining / {len(invalid_ids)} originally invalid; retained valid={entry["n"]-entry["invalid_n"]}')
    api_key = load_settings() if todo and transport is None else 'offline-test-key'
    if todo:
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            remaining = iter(todo)
            futures = {}
            for rid in list(todo)[:WORKERS]:
                next(remaining)
                futures[pool.submit(query_target, raw_map[rid], history.get(rid, []), log_path, api_key, transport)] = rid
            completed = 0
            while futures:
                done, _ = wait(futures, return_when=FIRST_COMPLETED)
                # Check every completed call for errors before scheduling more.
                for future in done:
                    future.result()
                for future in done:
                    futures.pop(future)
                    completed += 1
                    if completed % 25 == 0 or completed == len(todo):
                        print(f'  completed {completed}/{len(todo)} targets', flush=True)
                    rid = next(remaining, None)
                    if rid is not None:
                        futures[pool.submit(query_target, raw_map[rid], history.get(rid, []), log_path, api_key, transport)] = rid
    history = read_attempts(log_path)
    if set(history) != set(invalid_ids) or any(not history[rid][-1]['terminal'] for rid in invalid_ids):
        raise RuntimeError('Retry coverage incomplete; do not publish partial run')
    merged = []
    for old in original_raw:
        rid = old['record_id']
        if rid not in history:
            merged.append(old)
            continue
        last = history[rid][-1]
        row = {**old, 'raw_response': last['raw_response'], 'request_error': last['request_error'],
               'request_status': 'ok' if last['request_error'] is None else 'invalid_exhausted',
               'response_status': last['response_status'], 'attempts': len(history[rid]),
               'model': MODEL, 'provider': PROVIDER, 'finish_reason': last.get('finish_reason'),
               'usage': last.get('usage'), 'repair_original_model': old.get('model'),
               'repair_original_response_sha256': sha_bytes(old.get('raw_response', '').encode('utf-8')),
               'repair_plan_sha256': sha_file(WORK / 'plan.json'),
               'repair_attempt_log': str(log_path), 'repair_completed_at': last['finished_at'],
               'inference_fingerprint': sha_bytes(json.dumps(generation_spec(), sort_keys=True).encode('utf-8'))}
        merged.append(row)
    if len(merged) != entry['n'] or set(index_unique(merged, 'merged raw')) != set(raw_map):
        raise RuntimeError('Merged cohort/ID set changed')
    for row in merged:
        if row['record_id'] not in history and row != raw_map[row['record_id']]:
            raise RuntimeError('Previously valid response was modified')
    stage = task / 'staged_run'
    write_jsonl(stage / 'inference/raw_responses.jsonl', merged)
    labels = [{'record_id': r['record_id'], 'sealed_label': r['sealed_label']}
              for r in read_jsonl(snapshot / 'evaluation/predictions.jsonl')]
    write_jsonl(task / 'sealed_labels.jsonl', labels)
    metrics = evaluate(stage / 'inference/raw_responses.jsonl', task / 'sealed_labels.jsonl', stage / 'evaluation')
    model_counts = Counter(r.get('model', 'missing') for r in merged)
    manifest = {'schema': 'invalid_retry_publish_v1', 'run_id': run_id,
        'created_at': now(), 'source_snapshot': str(snapshot), 'source_fingerprints': entry['source_fingerprints'],
        'generation_spec': generation_spec(), 'originally_invalid_n': entry['invalid_n'],
        'retained_previously_valid_n': entry['n'] - entry['invalid_n'],
        'model_counts': dict(model_counts), 'mixed_backbone': True,
        'benchmark_comparable_single_backbone': False,
        'before': json.loads((snapshot / 'evaluation/evaluation.json').read_text(encoding='utf-8')),
        'after': metrics, 'attempts_path': str(log_path),
        'metadata_note': 'run_manifest.json/freeze.json describe original preparation; this repair manifest describes replacement inference and mixed model results'}
    write_json(stage / 'repair_manifest.json', manifest)
    publish_names = ('inference/raw_responses.jsonl', 'evaluation/predictions.jsonl',
                     'evaluation/evaluation.json', 'repair_manifest.json')
    published_hashes = {name: sha_file(stage / name) for name in publish_names}
    journal = task / 'publish_journal.json'
    previous = json.loads(journal.read_text(encoding='utf-8')) if journal.exists() else {}
    targets = [Path(entry['share']), Path(entry['original'])]
    accepted_hashes = {}
    for name in publish_names:
        known = set(previous.get('accepted_destination_hashes', {}).get(name, []))
        known.update((entry['source_fingerprints'].get(name), published_hashes[name],
                      previous.get('published_hashes', {}).get(name)))
        accepted_hashes[name] = sorted(value for value in known if value is not None)
    # Validate both destinations before the first replacement. Interrupted
    # publication is resumable from either original or previous staged hashes.
    for target in targets:
        for name in publish_names:
            path = target / name
            allowed = set(accepted_hashes[name])
            if path.exists() and sha_file(path) not in allowed:
                raise RuntimeError(f'Destination changed outside this repair: {path}')
        if sha_file(target / 'prompts/prompts.jsonl') != entry['source_fingerprints']['prompts/prompts.jsonl']:
            raise RuntimeError('Saved prompt corpus changed before publish')
    write_json(journal, {'phase': 'publishing', 'published_hashes': published_hashes,
                         'accepted_destination_hashes': accepted_hashes, 'updated_at': now()})
    for target in targets:
        for name in publish_names:
            atomic_bytes(target / name, (stage / name).read_bytes())
        # Keep ancillary summaries consistent if these exist in the original run.
        for name in ('completion.json', 'ablation_manifest.json'):
            path = target / name
            if path.exists():
                before_path = task / 'ancillary_originals' / ('share' if target == targets[0] else 'C') / name
                if not before_path.exists():
                    atomic_bytes(before_path, path.read_bytes())
                value = json.loads(path.read_text(encoding='utf-8'))
                value['evaluation'] = metrics
                value['repair_manifest'] = 'repair_manifest.json'
                value['mixed_backbone'] = True
                if name == 'completion.json':
                    value['request_error_count'] = sum(r.get('request_status') != 'ok' for r in merged)
                write_json(path, value)
    for target in targets:
        for name, expected in published_hashes.items():
            if sha_file(target / name) != expected:
                raise RuntimeError(f'Published checksum mismatch: {target/name}')
    write_json(journal, {'phase': 'complete', 'published_hashes': published_hashes,
                         'accepted_destination_hashes': accepted_hashes, 'updated_at': now()})
    print(f'BEFORE: valid={entry["n"]-entry["invalid_n"]}, invalid={entry["invalid_n"]}')
    print(f'AFTER:  valid={metrics["n_valid"]}, invalid={metrics["n_invalid"]}, coverage={metrics["coverage"]:.2%}')
    for name, result in [('merged pipeline', metrics), ('always vulnerable', metrics['always_vulnerable']),
                         ('always non-vulnerable', metrics['always_non_vulnerable'])]:
        print(name, {k: result[k] for k in ('confusion_matrix', 'accuracy_valid', 'precision', 'recall', 'f1', 'mcc', 'balanced_accuracy')})
    print('Updated SHARE and original C run. Mixed backbone: old Vertex/Qwen + repaired FPT/DeepSeek.')
    return manifest


PLAN = preflight()
print('Provider:', PROVIDER, '| model:', MODEL, '| context bytes:', CONTEXT_BYTES)
print('Run the six branch cells below. No API call is made by this setup cell.')


## 1. Devign / Plus-C8

Originally invalid: **1251/2732 = 45.79%**. Cell này tự đánh giá lại toàn bộ 2732 mẫu sau khi ghép.


In [ ]:
result_1 = run_branch('Devign/Ablations/Plus-C8-mmr-selection/test_v2')


## 2. Reveal / No-C2

Originally invalid: **1013/2274 = 44.55%**. Cell này tự đánh giá lại toàn bộ 2274 mẫu sau khi ghép.


In [ ]:
result_2 = run_branch('Reveal/Ablations/No-C2-stratified-demo-selection/test_v2')


## 3. Reveal / No-C3

Originally invalid: **930/2274 = 40.90%**. Cell này tự đánh giá lại toàn bộ 2274 mẫu sau khi ghép.


In [ ]:
result_3 = run_branch('Reveal/Ablations/No-C3-GEPA-prompt-optimization/test_v2')


## 4. Reveal / No-C5

Originally invalid: **1036/2274 = 45.56%**. Cell này tự đánh giá lại toàn bộ 2274 mẫu sau khi ghép.


In [ ]:
result_4 = run_branch('Reveal/Ablations/No-C5-graph-verbalization/test_v2')


## 5. Reveal / Plus-C1

Originally invalid: **934/2274 = 41.07%**. Cell này tự đánh giá lại toàn bộ 2274 mẫu sau khi ghép.


In [ ]:
result_5 = run_branch('Reveal/Ablations/Plus-C1-tri-signal-retrieval/test_v2')


## 6. Reveal / Plus-C7

Originally invalid: **1158/2274 = 50.92%**. Cell này tự đánh giá lại toàn bộ 2274 mẫu sau khi ghép.


In [ ]:
result_6 = run_branch('Reveal/Ablations/Plus-C7-security-cot/test_v2')


## Tổng hợp sau các nhánh

Cell dưới chỉ đọc evaluation mới; không gọi LLM.


In [ ]:
summary = []
for run_id in BRANCHES:
    journal_path = WORK / 'repairs' / run_id / 'publish_journal.json'
    if not journal_path.exists() or json.loads(journal_path.read_text(encoding='utf-8'))['phase'] != 'complete':
        print('NOT PUBLISHED:', run_id)
        continue
    report = json.loads((SHARE / 'runs' / run_id / 'repair_manifest.json').read_text(encoding='utf-8'))
    m = report['after']
    row = {'run_id': run_id, 'total': m['n_responses'], 'before_invalid': report['originally_invalid_n'],
           'after_invalid': m['n_invalid'], 'coverage': m['coverage'], 'accuracy': m['accuracy_valid'],
           'f1': m['f1'], 'mcc': m['mcc'], 'balanced_accuracy': m['balanced_accuracy'], 'mixed_backbone': True}
    summary.append(row)
    print(row)
write_json(WORK / 'summary.json', summary)
